# Generador sintético Bronze: CUMST, DEAL y TRANS

Simula una extracción del AS400 y la publica en `raw_data.bronze`. Tiene **dos modos**:

| Modo | Qué hace | Escritura | Filas por defecto (cumst / deal / trans) | Quién y cuándo |
|---|---|---|---|---|
| `inicial` | Reinicia Bronze desde cero con IDs `CUS-00001`, `DEA-00001`, `TRX-1000000001`... | `overwrite` | 500 / 500 / 1,500 | **Solo el instructor**, antes del taller |
| `incremental` | Agrega un **lote nuevo** que continúa los IDs desde el último existente | `append` | 50 / 50 / 150 | Instructor (o el Job) en el Paso 18 |

**Reglas importantes**
- Bronze es **compartido** por todos los ingenieros: ellos **no** ejecutan este notebook, solo leen las tablas.
- El modo `inicial` **reescribe** las tablas. Las Streaming Tables del pipeline NRT fallan si su fuente se reescribe, así que después de un `inicial` hay que hacer **Full refresh** de los pipelines NRT que ya existan.
- En cada lote, los préstamos se asignan a clientes del mismo lote y cada transacción toma un préstamo y su cliente: las FKs siempre son válidas.
- `multiplicador_espera = 0` ejecuta sin pausas (≈1 min). Con `1` simula una extracción lenta de ≈30 min.

Los parámetros se cambian en los widgets de arriba o desde un Job (`base_parameters`).

In [0]:
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F

# ---------- Widgets ----------
dbutils.widgets.dropdown("modo", "inicial", ["inicial", "incremental"], "1. Modo de carga")
dbutils.widgets.text("filas_cumst", "", "2. Filas CUMST (vacío = automático)")
dbutils.widgets.text("filas_deal", "", "3. Filas DEAL (vacío = automático)")
dbutils.widgets.text("filas_trans", "", "4. Filas TRANS (vacío = automático)")
dbutils.widgets.text("multiplicador_espera", "0", "5. Multiplicador de espera (0 = sin pausas)")
dbutils.widgets.text("catalogo", "raw_data", "6. Catálogo Bronze")
dbutils.widgets.text("esquema", "bronze", "7. Esquema Bronze")

MODO = dbutils.widgets.get("modo").strip().lower()
if MODO not in ("inicial", "incremental"):
    raise ValueError(f"Modo no válido: {MODO}. Usa 'inicial' o 'incremental'.")

# Filas por defecto según el modo (si el widget queda vacío)
DEFAULT_ROWS = {"inicial": (500, 500, 1500), "incremental": (50, 50, 150)}

def leer_entero(nombre, defecto):
    valor = dbutils.widgets.get(nombre).strip()
    return int(valor) if valor else defecto

CUMST_ROWS = leer_entero("filas_cumst", DEFAULT_ROWS[MODO][0])
DEAL_ROWS = leer_entero("filas_deal", DEFAULT_ROWS[MODO][1])
TRANS_ROWS = leer_entero("filas_trans", DEFAULT_ROWS[MODO][2])
if min(CUMST_ROWS, DEAL_ROWS, TRANS_ROWS) < 1:
    raise ValueError("Las filas de cumst, deal y trans deben ser mayores que 0.")

RUNTIME_MULTIPLIER = float(dbutils.widgets.get("multiplicador_espera").strip() or 0)
CATALOG_NAME = dbutils.widgets.get("catalogo").strip()
SCHEMA_NAME = dbutils.widgets.get("esquema").strip()
SOURCE_SYSTEM = "AS400_GENERIC"

RUN_TS = datetime.now(timezone.utc)
BATCH_ID = f"{'INI' if MODO == 'inicial' else 'INC'}-{RUN_TS.strftime('%Y%m%d%H%M%S')}-{uuid.uuid4().hex[:6].upper()}"

# Minutos simulados por etapa (solo aplican si multiplicador_espera > 0)
PIPELINE_STAGE_PLAN = {"catalog": 5, "generate": 15, "publish": 7, "validation": 3}

print(f"Modo:      {MODO}")
print(f"Destino:   {CATALOG_NAME}.{SCHEMA_NAME}")
print(f"Filas:     cumst={CUMST_ROWS} | deal={DEAL_ROWS} | trans={TRANS_ROWS}")
print(f"Espera:    multiplicador={RUNTIME_MULTIPLIER}")
print(f"Batch ID:  {BATCH_ID}")

In [0]:
def run_stage(stage_name, duration_minutes, fn=None):
    """Ejecuta una etapa. Si multiplicador_espera > 0 simula su duración con pausas."""
    total_seconds = int(duration_minutes * 60 * RUNTIME_MULTIPLIER)
    print(f"\n=== {stage_name} ===")
    elapsed = 0
    while elapsed < total_seconds:
        sleep_for = min(15, total_seconds - elapsed)
        time.sleep(sleep_for)
        elapsed += sleep_for
        print(f"[{stage_name}] {round(elapsed / total_seconds * 100, 1)}% completado")
    return fn() if fn is not None else None


def tabla(nombre):
    return f"{CATALOG_NAME}.{SCHEMA_NAME}.{nombre}"


def ultimo_numero(nombre_tabla, columna):
    """Devuelve la parte numérica más alta de un ID (ej. CUS-00550 -> 550)."""
    return (
        spark.table(nombre_tabla)
        .select(F.max(F.regexp_extract(columna, r"(\d+)$", 1).cast("long")))
        .first()[0]
    ) or 0


def preparar_destino():
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
    if MODO == "inicial":
        print("Modo inicial: los IDs empiezan desde 1.")
        return 0, 0, 0
    faltantes = [t for t in ("cumst", "deal", "trans") if not spark.catalog.tableExists(tabla(t))]
    if faltantes:
        raise ValueError(f"No existen {faltantes} en {CATALOG_NAME}.{SCHEMA_NAME}. Ejecuta primero el modo 'inicial'.")
    off_c = ultimo_numero(tabla("cumst"), "customer_id")
    off_d = ultimo_numero(tabla("deal"), "deal_id")
    off_t = ultimo_numero(tabla("trans"), "trx_id") - 1000000000   # TRX-1000000001 es la primera
    print(f"Modo incremental: el lote continúa después de CUS-{off_c:05d}, DEA-{off_d:05d}, TRX-{off_t + 1000000000:010d}")
    return off_c, off_d, off_t


OFF_CUMST, OFF_DEAL, OFF_TRANS = run_stage("1/4 - Preparar catálogo, esquema y offsets", PIPELINE_STAGE_PLAN["catalog"], preparar_destino)

In [0]:
def build_cumst(n_rows, offset):
    male_names = F.array(*[F.lit(x) for x in ["Eirik", "Leif", "Bjorn", "Andreas", "Soren", "Magnus"]])
    female_names = F.array(*[F.lit(x) for x in ["Astrid", "Freja", "Ingrid", "Sofia", "Eleni", "Mariam"]])
    nordic_last = F.array(*[F.lit(x) for x in ["Johansen", "Andersson", "Nielsen", "Svensson", "Larsen", "Hansen"]])
    greek_last = F.array(*[F.lit(x) for x in ["Papadopoulos", "Nikolaidis", "Georgiou", "Kostopoulos", "Vassiliou"]])
    palestinian_last = F.array(*[F.lit(x) for x in ["Al Khalidi", "Hamdan", "Barghouti", "Abu Salem", "Nusseibeh"]])
    no_cities = F.array(*[F.lit(x) for x in ["Oslo", "Bergen", "Trondheim"]])
    gr_cities = F.array(*[F.lit(x) for x in ["Athens", "Thessaloniki", "Patras"]])
    ps_cities = F.array(*[F.lit(x) for x in ["Ramallah", "Nablus", "Bethlehem"]])
    # row_id global: continúa después del último cliente existente
    df = spark.range(offset, offset + n_rows).withColumnRenamed("id", "row_id")
    gender = F.when((F.col("row_id") % 2) == 0, F.lit("M")).otherwise(F.lit("F"))
    country = (F.when((F.col("row_id") % 3) == 0, F.lit("NO"))
                .when((F.col("row_id") % 3) == 1, F.lit("GR"))
                .otherwise(F.lit("PS")))
    first_name = F.when(gender == "M", F.element_at(male_names, (F.pmod(F.col("row_id"), F.lit(6)) + F.lit(1)).cast("int"))
                ).otherwise(F.element_at(female_names, (F.pmod(F.col("row_id"), F.lit(6)) + F.lit(1)).cast("int")))
    last_name = (F.when(country == "NO", F.element_at(nordic_last, (F.pmod(F.col("row_id"), F.lit(6)) + F.lit(1)).cast("int")))
                  .when(country == "GR", F.element_at(greek_last, (F.pmod(F.col("row_id"), F.lit(5)) + F.lit(1)).cast("int")))
                  .otherwise(F.element_at(palestinian_last, (F.pmod(F.col("row_id"), F.lit(5)) + F.lit(1)).cast("int"))))
    city = (F.when(country == "NO", F.element_at(no_cities, (F.pmod(F.col("row_id"), F.lit(3)) + F.lit(1)).cast("int")))
             .when(country == "GR", F.element_at(gr_cities, (F.pmod(F.col("row_id"), F.lit(3)) + F.lit(1)).cast("int")))
             .otherwise(F.element_at(ps_cities, (F.pmod(F.col("row_id"), F.lit(3)) + F.lit(1)).cast("int"))))
    email_domain = (F.when(country == "NO", F.lit("nordmail.com"))
                     .when(country == "GR", F.lit("hellasmail.gr"))
                     .otherwise(F.lit("levantmail.ps")))
    doc_prefix = F.when(country == "NO", F.lit("NOR")).when(country == "GR", F.lit("GRC")).otherwise(F.lit("PAL"))
    phone_prefix = F.when(country == "NO", F.lit("+47")).when(country == "GR", F.lit("+30")).otherwise(F.lit("+970"))
    return (df
        .withColumn("customer_id", F.format_string("CUS-%05d", F.col("row_id") + F.lit(1)))
        .withColumn("gender", gender)
        .withColumn("first_name", first_name)
        .withColumn("last_name", last_name)
        .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
        .withColumn("country", country)
        .withColumn("city", city)
        .withColumn("document_id", F.concat(doc_prefix, F.lit("-"), F.format_string("%09d", F.col("row_id") + F.lit(100000001))))
        .withColumn("email", F.lower(F.concat_ws(".", F.regexp_replace(F.col("first_name"), " ", ""), F.regexp_replace(F.col("last_name"), " ", ""), email_domain)))
        .withColumn("phone", F.concat(phone_prefix, F.lit("-"), F.format_string("%08d", F.col("row_id") + F.lit(10000000))))
        .withColumn("birth_date", F.date_sub(F.current_date(), (F.lit(18 * 365) + (F.col("row_id") % F.lit(45)) * F.lit(365)).cast("int")))
        .withColumn("status", F.when((F.col("row_id") % 10) == 0, F.lit("I")).otherwise(F.lit("A")))
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("ingest_ts", F.lit(RUN_TS))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingest_date", F.to_date(F.lit(RUN_TS.date().isoformat())))
        .drop("row_id"))


def build_deal(cumst_df, n_rows, offset):
    n_clientes = cumst_df.count()
    # cada préstamo se asigna a un cliente del MISMO lote (así la FK siempre es válida)
    customer_lookup = cumst_df.select("customer_id", F.row_number().over(Window.orderBy("customer_id")).alias("customer_rn"))
    base = (spark.range(offset, offset + n_rows)
            .withColumn("rn", (F.pmod(F.col("id") - F.lit(offset), F.lit(n_clientes)) + F.lit(1)).cast("int")))
    product_code = F.array(*[F.lit(x) for x in ["LOAN", "CARD", "MORT", "SAV", "INV", "AUTO"]])
    currency = F.when((F.col("id") % 5) == 0, F.lit("USD")).otherwise(F.lit("CRC"))
    return (base
        .join(F.broadcast(customer_lookup), F.col("rn") == F.col("customer_rn"), "left")
        .withColumn("deal_id", F.format_string("DEA-%05d", F.col("id") + F.lit(1)))
        .withColumn("product_code", F.element_at(product_code, (F.pmod(F.col("id"), F.lit(6)) + F.lit(1)).cast("int")))
        .withColumn("currency", currency)
        .withColumn("principal_balance", F.round((F.pow((F.col("id") % 97) + F.lit(2), F.lit(1.45)) * F.lit(123.45)).cast("double"), 2))
        .withColumn("interest_rate", F.round(F.lit(0.08) + (F.col("id") % F.lit(40)) * F.lit(0.0015), 6))
        .withColumn("open_date", F.date_sub(F.current_date(), (F.col("id") % F.lit(720)).cast("int")))
        .withColumn("maturity_date", F.date_add(F.current_date(), (F.col("id") % F.lit(900)).cast("int")))
        .withColumn("status", F.when((F.col("id") % 12) == 0, F.lit("S")).otherwise(F.lit("A")))
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("ingest_ts", F.lit(RUN_TS))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingest_date", F.to_date(F.lit(RUN_TS.date().isoformat())))
        .select("customer_id", "deal_id", "product_code", "currency", "principal_balance", "interest_rate",
                "open_date", "maturity_date", "status", "source_system", "ingest_ts", "batch_id", "ingest_date"))


def build_trans(deal_df, n_rows, offset):
    n_prestamos = deal_df.count()
    # cada transacción toma un préstamo del MISMO lote y el cliente dueño de ese préstamo
    deal_lookup = deal_df.select(
        F.col("deal_id").alias("deal_id_key"),
        F.col("customer_id").alias("deal_customer_id"),
        F.row_number().over(Window.orderBy("deal_id")).alias("deal_rn"))
    trx_codes = F.array(*[F.lit(x) for x in ["DEP", "RET", "TRF", "INT", "COM", "PAY", "REV"]])
    base = (spark.range(offset, offset + n_rows)
            .withColumn("row_id", (F.col("id") + F.lit(1)).cast("long"))
            .withColumn("deal_rn", (F.pmod(F.col("id") - F.lit(offset), F.lit(n_prestamos)) + F.lit(1)).cast("int")))
    return (base
        .join(F.broadcast(deal_lookup), "deal_rn", "left")
        .withColumn("trx_id", F.format_string("TRX-%010d", F.col("row_id") + F.lit(1000000000)))
        .withColumn("tracde", F.element_at(trx_codes, (F.pmod(F.col("id"), F.lit(7)) + F.lit(1)).cast("int")))
        .withColumn("tradcc", F.when(F.col("tracde").isin("RET", "COM", "PAY"), F.lit("D")).otherwise(F.lit("C")))
        .withColumn("customer_id", F.col("deal_customer_id"))
        .withColumn("deal_id", F.col("deal_id_key"))
        .withColumn("tracun", F.col("deal_customer_id"))
        .withColumn("tragln", F.regexp_replace(F.col("deal_id_key"), "DEA-", "").cast("long") + F.lit(10000000000))
        .withColumn("traacc", F.regexp_replace(F.col("deal_id_key"), "DEA-", "").cast("long") + F.lit(7000000000))
        .withColumn("trabnk", F.lit("001"))
        .withColumn("trabrn", (F.pmod(F.col("id"), F.lit(90)) + F.lit(1)).cast("int"))
        .withColumn("traccy", F.when((F.col("id") % 5) == 0, F.lit("USD")).otherwise(F.lit("CRC")))
        .withColumn("traamt", F.round((F.pow((F.pmod(F.col("id"), F.lit(180)) + F.lit(2)).cast("double"), F.lit(1.18)) * F.lit(17.35)), 2))
        .withColumn("travdm", F.date_add(F.current_date(), -(F.pmod(F.col("id"), F.lit(120))).cast("int")))
        .withColumn("trabdy", F.date_sub(F.current_date(), (F.pmod(F.col("id"), F.lit(120))).cast("int")))
        .withColumn("trausr", F.concat(F.lit("USR"), F.format_string("%03d", F.pmod(F.col("id"), F.lit(75)) + F.lit(1))))
        .withColumn("traapc", F.lit("AS400"))
        .withColumn("tramod", F.when((F.col("id") % 3) == 0, F.lit("L")).otherwise(F.lit("T")))
        .withColumn("trapmn", F.pmod(F.col("id"), F.lit(12)) + F.lit(1))
        .withColumn("trackn", F.pmod(F.col("id"), F.lit(999999)) + F.lit(1))
        .withColumn("traccn", (F.pmod(F.col("id"), F.lit(25)) + F.lit(100)).cast("int"))
        .withColumn("source_system", F.lit(SOURCE_SYSTEM))
        .withColumn("ingest_ts", F.lit(RUN_TS))
        .withColumn("batch_id", F.lit(BATCH_ID))
        .withColumn("ingest_date", F.to_date(F.lit(RUN_TS.date().isoformat())))
        .drop("id", "row_id", "deal_rn", "deal_id_key", "deal_customer_id"))

In [0]:
def generar_lote():
    cumst_df = build_cumst(CUMST_ROWS, OFF_CUMST)
    deal_df = build_deal(cumst_df, DEAL_ROWS, OFF_DEAL)
    trans_df = build_trans(deal_df, TRANS_ROWS, OFF_TRANS)
    print("Vista previa del lote:")
    cumst_df.select("customer_id", "full_name", "country", "city").show(3, truncate=False)
    deal_df.select("deal_id", "customer_id", "currency", "principal_balance").show(3, truncate=False)
    trans_df.select("trx_id", "customer_id", "deal_id", "tracde", "traamt", "travdm").show(3, truncate=False)
    return cumst_df, deal_df, trans_df

cumst_df, deal_df, trans_df = run_stage("2/4 - Generar CUMST, DEAL y TRANS", PIPELINE_STAGE_PLAN["generate"], generar_lote)

## Publicar en Bronze

- `inicial` → `overwrite`: reemplaza las tres tablas.
- `incremental` → `append`: agrega el lote al final. Se escribe primero `cumst`, luego `deal` y al final `trans`, para que las FKs existan antes que las transacciones.

In [0]:
def publicar():
    modo_escritura = "overwrite" if MODO == "inicial" else "append"
    for nombre, df in (("cumst", cumst_df), ("deal", deal_df), ("trans", trans_df)):
        writer = df.write.format("delta").mode(modo_escritura)
        if MODO == "inicial":
            writer = writer.option("overwriteSchema", "true")
        writer.saveAsTable(tabla(nombre))
        print(f"{modo_escritura:9} -> {tabla(nombre)}")

run_stage("3/4 - Publicar en Bronze", PIPELINE_STAGE_PLAN["publish"], publicar)

## Validación

Revisa conteos, duplicados y FKs huérfanas. Si algo falla, el notebook se detiene con error (y un Job quedaría en rojo).

In [0]:
def validar():
    cumst, deal, trans = (spark.table(tabla(t)) for t in ("cumst", "deal", "trans"))
    resultados = {
        "filas cumst": cumst.count(),
        "filas deal": deal.count(),
        "filas trans": trans.count(),
        "duplicados cumst": cumst.groupBy("customer_id").count().filter("count > 1").count(),
        "duplicados deal": deal.groupBy("deal_id").count().filter("count > 1").count(),
        "duplicados trans": trans.groupBy("trx_id").count().filter("count > 1").count(),
        "deal sin cliente": deal.join(cumst, "customer_id", "left_anti").count(),
        "trans sin cliente": trans.join(cumst, "customer_id", "left_anti").count(),
        "trans sin prestamo": trans.join(deal, "deal_id", "left_anti").count(),
    }
    for k, v in resultados.items():
        print(f"{k:20}: {v}")

    print("\nFilas por lote (batch_id):")
    (trans.groupBy("batch_id").agg(F.count("*").alias("filas_trans"), F.min("ingest_ts").alias("ingest_ts"))
          .orderBy("ingest_ts").show(truncate=False))

    errores = {k: v for k, v in resultados.items() if not k.startswith("filas") and v > 0}
    if errores:
        raise ValueError(f"Validación de Bronze falló: {errores}")
    print("✅ Bronze válido")
    if MODO == "inicial":
        print("⚠️  Bronze se reinició: haz FULL REFRESH de los pipelines NRT que ya existan antes de volver a ejecutarlos.")

run_stage("4/4 - Validar Bronze", PIPELINE_STAGE_PLAN["validation"], validar)

## Siguientes pasos

- **Después de `inicial`:** Bronze queda en 500 / 500 / 1,500. Los ingenieros crean y ejecutan sus pipelines Batch y NRT.
- **Después de `incremental`:** cada ingeniero vuelve a ejecutar su pipeline NRT. Las Streaming Tables procesan **solo** las filas del nuevo lote y Gold se recalcula.
- **Automatizar los lotes:** el Job `Lab_BAC_Bronze_Lote_Incremental` corre este notebook en modo `incremental` y luego actualiza el pipeline NRT. Su programación (cada 5 min) viene **pausada**: actívala solo durante el bloque NRT y vuelve a pausarla al terminar.